# Scenario: Customer Support Chatbot Workflow
 Imagine a company wants to build a chatbot that helps customers with quick answers. The workflow is modeled as a graph of states:

 - State Definition (BotState)
 - The chatbot keeps track of:
 - The question asked by the customer.
 - The answer generated.
 - The history of all past questions.
 - Think of this as the chatbot’s notebook where it records the conversation.

 - Nodes (Functions)
 - get_answer:
 When a customer asks, “What are your store hours?”, the chatbot looks at the question and generates a placeholder answer like “Answer to: What are your store hours?”.
 It also adds the question to the history log.
 - format_output:
 Before sending the response back to the customer, the chatbot reformats it into a friendly style:
 “Bot says: Answer to: What are your store hours?”

 - Graph Flow
 - The chatbot starts at the get_answer node (entry point).
 - Once the answer is generated, it flows to the format_output node.
 - Finally, the conversation ends at END, meaning the chatbot has
  delivered its response.


In [1]:


from langgraph.graph import StateGraph, END
from typing import TypedDict

# 1. Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# 2. Define Nodes (functions)
def get_answer(state: BotState):
    q = state["question"]
    # In real app: call LLM here
    ans = f"Answer to: {q}"
    return {"answer": ans,
            "history": state["history"] + [q]}

def format_output(state: BotState):
    return {"answer": f"Bot says: {state['answer']}"}

# 3. Build the Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

# 4. Add Edges
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)


In [4]:
# Step 1: Install huggingface-hub if not already installed
!pip install huggingface-hub --quiet

# Step 2: Import libraries
from huggingface_hub import InferenceClient
from getpass import getpass

# Step 3: Securely input Hugging Face API key
api_key = getpass("Enter your Hugging Face API key (hidden): ")

# Step 4: Initialize Hugging Face Inference Client
client = InferenceClient(api_key=api_key)

# ====================================
# 🧠 SHARED MEMORY (simple dict)
# ====================================
memory = {}

# ====================================
# 🤖 LLM CALL
# ====================================
def call_llm(prompt):
    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"ERROR: {str(e)}"

# ====================================
# 🔍 SEARCH AGENT
# ====================================
def search_agent(goal):
    print("\n[Search Agent] Working...")
    result = call_llm(f"Find key data and facts about: {goal}")
    memory["search"] = result
    return result

# ====================================
# 📊 ANALYST AGENT
# ====================================
def analyst_agent(goal):
    print("\n[Analyst Agent] Working...")
    result = call_llm(f"""
Goal: {goal}

Data:
{memory.get("search")}

Analyze trends and insights.
""")
    memory["analyst"] = result
    return result

# ====================================
# ✍️ WRITER AGENT
# ====================================
def writer_agent(goal):
    print("\n[Writer Agent] Working...")
    result = call_llm(f"""
Goal: {goal}

Insights:
{memory.get("analyst")}

Write a structured report.
""")
    memory["writer"] = result
    return result

# ====================================
# ✅ QA AGENT
# ====================================
def qa_agent(goal):
    print("\n[QA Agent] Working...")
    result = call_llm(f"""
Goal: {goal}

Report:
{memory.get("writer")}

Improve quality and fix errors.
""")
    memory["qa"] = result
    return result

# ====================================
# 🚀 ORCHESTRATOR
# ====================================
def run_system(user_goal):
    print("\n🎯 Goal:", user_goal)

    search_agent(user_goal)
    analyst_agent(user_goal)
    writer_agent(user_goal)
    qa_agent(user_goal)

    return memory.get("qa")

# ====================================
# 🧑‍💻 USER INPUT LOOP
# ====================================
while True:
    user_input = input("\nEnter your goal (or 'exit'): ")

    if user_input.lower() == "exit":
        break

    final_output = run_system(user_input)
    print("\n✅ FINAL OUTPUT:\n", final_output)

Enter your Hugging Face API key (hidden): ··········

Enter your goal (or 'exit'): User Goal: "Write a report on AI trends in 2026"

🎯 Goal: User Goal: "Write a report on AI trends in 2026"

[Search Agent] Working...

[Analyst Agent] Working...

[Writer Agent] Working...

[QA Agent] Working...

✅ FINAL OUTPUT:
 **Report on AI Trends in 2026**

**Executive Summary**

This report provides insights into the future of Artificial Intelligence (AI) in 2026, highlighting key trends, technological advancements, and societal implications. As AI continues to transform industries and societies, it is essential to anticipate and prepare for the opportunities and challenges that lie ahead.

**Introduction**

Artificial Intelligence (AI) has been rapidly evolving over the past decade, with significant advancements in machine learning, natural language processing, and computer vision. As we look ahead to 2026, we can anticipate even more profound changes in the field of AI. This report provides an ov

# Scenario: Customer Support Chatbot (Question-Based)
Imagine a company has deployed a chatbot that answers customer
 questions by calling the Groq API. The workflow is modeled as a
  graph of states, where each customer query flows through nodes until
   a response is delivered.

 1. State Definition
 The chatbot maintains a notebook-like state:
 - question → The customer’s query.
 - answer → The response generated by Groq.
 - history → A log of all past questions.

In [14]:
# ====================================
# 🚀 STEP 1: Install dependencies
# ====================================
!pip install langgraph huggingface-hub --quiet

# ====================================
# 📦 STEP 2: Imports
# ====================================
from langgraph.graph import StateGraph, END
from typing import TypedDict
from huggingface_hub import InferenceClient
from getpass import getpass

# ====================================
# 🔐 STEP 3: API KEY
# ====================================
api_key = getpass("Enter your Hugging Face API key: ")
client = InferenceClient(api_key=api_key)

print("✅ API Loaded Successfully!")

# ====================================
# 🧠 STEP 4: DEFINE STATE
# ====================================
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# ====================================
# 🤖 STEP 5: NODE 1 (LLM CALL)
# ====================================
def get_answer(state: BotState):
    q = state["question"]

    try:
        print("\n⏳ Calling Hugging Face...")

        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[{"role": "user", "content": q}],
            max_tokens=100
        )

        ans = response.choices[0].message.content.strip()
        print("✅ Response received")

    except Exception as e:
        ans = f"Error: {str(e)}"
        print("❌ ERROR:", e)

    return {
        "answer": ans,
        "history": state["history"] + [q]
    }

# ====================================
# 🎨 STEP 6: NODE 2 (FORMAT OUTPUT)
# ====================================
def format_output(state: BotState):
    return {
        "answer": f"🤖 Bot says: {state['answer']}"
    }

# ====================================
# 🔗 STEP 7: BUILD GRAPH
# ====================================
graph = StateGraph(BotState)

graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)

# Compile graph
app = graph.compile()

# ====================================
# 🧪 STEP 8: RUN (INTERACTIVE)
# ====================================
print("\n🚀 Chatbot Started! Type 'exit' to stop.\n")

state = {"question": "", "answer": "", "history": []}

while True:
    user_q = input("You: ")

    if user_q.lower() == "exit":
        print("👋 Exiting...")
        break

    state["question"] = user_q

    result = app.invoke(state)

    print(result["answer"])

Enter your Hugging Face API key: ··········
✅ API Loaded Successfully!

🚀 Chatbot Started! Type 'exit' to stop.

You: What is AI?

⏳ Calling Hugging Face...
✅ Response received
🤖 Bot says: AI, or Artificial Intelligence, refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. The term can refer to any machine that exhibits traits associated with a human mind such as:

1. **Learning**: The ability to improve performance on a task through experience.
2. **Problem-solving**: The ability to identify and solve problems, often by using logic and reasoning.
3. **Reasoning**: The ability to make decisions based on data and logic.
4. **
You: exit
👋 Exiting...


# Scenario: Customer Support Call Center
 A company runs a support chatbot that needs to route customer queries to the right department. Instead of one big script, they design a state graph where each node represents a specialized agent.

 1. State Definition (SupportState)
 The chatbot keeps track of:
 - query → What the customer asked.
 - category → Which department it belongs to (billing, tech, general).
 - response → What the bot replies with.
 Think of this as the customer’s “ticket form.”

 2. Router Node (route_query)
 When a customer types a question, the router scans it:
 - If the query mentions “bill” or “payment”, it routes to billing_agent.
 - If it mentions “error” or “bug”, it routes to tech_agent.
 - Otherwise, it defaults to general_agent.
 This is like a receptionist deciding which desk you should go to.

 3. Agent Nodes
 - billing_agent → Replies with “Billing dept: [query]”.
 - tech_agent → Replies with “Tech support: [query]”.
 - general_agent → Replies with “General help: [query]”.
 Each agent specializes in its own type of problem.

 4. Graph Flow
 - Entry point: router.
 - Router decides the path based on the query.
 - The query flows into the correct agent node.
 - The agent generates a response and ends the conversation.

In [15]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal

class SupportState(TypedDict):
    query: str
    category: str   # "billing" | "tech" | "general"
    response: str

# Router: reads state, returns next node name
def route_query(state: SupportState) -> str:
    q = state["query"].lower()
    if "bill" in q or "payment" in q:
        return "billing_agent"
    elif "error" in q or "bug" in q:
        return "tech_agent"
    else:
        return "general_agent"

def billing_agent(state):
    return {"response": "Billing dept: " + state["query"]}

def tech_agent(state):
    return {"response": "Tech support: " + state["query"]}

def general_agent(state):
    return {"response": "General help: " + state["query"]}

# Build graph with conditional routing
g = StateGraph(SupportState)
g.add_node("billing_agent", billing_agent)
g.add_node("tech_agent", tech_agent)
g.add_node("general_agent", general_agent)

# One entry point routes to 3 different nodes!
g.set_entry_point("router")
g.add_conditional_edges(
    "router",    # from node
    route_query, # function that returns next node
    {            # mapping: return value → node name
        "billing_agent":  "billing_agent",
        "tech_agent":     "tech_agent",
        "general_agent":  "general_agent",
    }
)


**Quantum Computing**

In [16]:
# ====================================
# 🚀 STEP 1: Install
# ====================================
!pip install langgraph huggingface-hub --quiet

# ====================================
# 📦 STEP 2: Imports
# ====================================
from langgraph.graph import StateGraph, END
from typing import TypedDict
from huggingface_hub import InferenceClient
from getpass import getpass

# ====================================
# 🔐 STEP 3: API KEY
# ====================================
api_key = getpass("Enter your Hugging Face API key: ")
client = InferenceClient(api_key=api_key)

print("✅ API Loaded!")

# ====================================
# 🧠 STATE
# ====================================
class ResearchState(TypedDict):
    topic: str
    search_results: list
    analysis: str
    summary: str
    steps_done: int

# ====================================
# 🤖 LLM CALL
# ====================================
def call_llm(prompt):
    try:
        print("⏳ LLM call...")
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=150
        )
        print("✅ Done")
        return response.choices[0].message.content.strip()
    except Exception as e:
        print("❌ ERROR:", e)
        return "Error generating response"

# ====================================
# 🔍 NODE 1: SEARCH (LLM-based)
# ====================================
def search_web(state: ResearchState):
    print(f"\n🔍 Searching: {state['topic']}")

    prompt = f"""
    Give 2 short factual points about: {state['topic']}.
    Each point should be 1 line.
    """

    result = call_llm(prompt)

    new_results = result.split("\n")

    return {
        "search_results": state["search_results"] + new_results,
        "steps_done": state["steps_done"] + 1
    }

# ====================================
# 🧠 NODE 2: ANALYZE
# ====================================
def analyze_results(state: ResearchState):
    print(f"\n🧠 Analyzing {len(state['search_results'])} results")

    prompt = f"""
    Topic: {state['topic']}

    Data:
    {state['search_results']}

    Extract key insights in 2-3 lines.
    """

    analysis = call_llm(prompt)

    return {
        "analysis": analysis,
        "steps_done": state["steps_done"] + 1
    }

# ====================================
# 📄 NODE 3: SUMMARIZE
# ====================================
def summarize(state: ResearchState):
    print("\n📄 Generating summary...")

    prompt = f"""
    Topic: {state['topic']}
    Insights: {state['analysis']}

    Give a short final summary.
    """

    summary = call_llm(prompt)

    return {"summary": summary}

# ====================================
# 🔁 NODE 4: CONDITION
# ====================================
def should_continue(state: ResearchState):
    if len(state["search_results"]) < 4:
        return "search_web"
    return END

# ====================================
# 🔗 BUILD GRAPH
# ====================================
g = StateGraph(ResearchState)

g.add_node("search_web", search_web)
g.add_node("analyze", analyze_results)
g.add_node("summarize", summarize)

g.set_entry_point("search_web")

g.add_edge("search_web", "analyze")

g.add_conditional_edges(
    "analyze",
    should_continue,
    {
        "search_web": "search_web",
        END: "summarize"
    }
)

g.add_edge("summarize", END)

app = g.compile()

# ====================================
# ▶️ RUN
# ====================================
result = app.invoke(
    {
        "topic": "Quantum Computing",
        "search_results": [],
        "analysis": "",
        "summary": "",
        "steps_done": 0
    }
)

print("\n✅ FINAL SUMMARY:\n")
print(result["summary"])

Enter your Hugging Face API key: ··········
✅ API Loaded!

🔍 Searching: Quantum Computing
⏳ LLM call...
✅ Done

🧠 Analyzing 4 results
⏳ LLM call...
✅ Done

📄 Generating summary...
⏳ LLM call...
✅ Done

✅ FINAL SUMMARY:

**Summary:** Quantum Computing offers a revolutionary approach to processing information, enabling faster calculations and parallel processing capabilities through the use of quantum bits (qubits). By harnessing the principles of superposition and entanglement, quantum computers can perform exponentially faster calculations than their classical counterparts, opening up new possibilities for complex problem-solving and data processing.


**Quantum Computing with API**

In [18]:
# ====================================
# 🚀 STEP 1: Install dependencies
# ====================================
!pip install langgraph huggingface-hub --quiet

# ====================================
# 📦 STEP 2: Imports
# ====================================
from langgraph.graph import StateGraph, END
from typing import TypedDict
from huggingface_hub import InferenceClient
from getpass import getpass

# ====================================
# 🔐 STEP 3: API KEY
# ====================================
api_key = getpass("Enter your Hugging Face API key: ")
client = InferenceClient(api_key=api_key)

print("✅ API Loaded!")

# ====================================
# 🧠 STATE
# ====================================
class ResearchState(TypedDict):
    topic: str
    search_results: list
    analysis: str
    summary: str
    steps_done: int

# ====================================
# 🤖 HELPER: HF API CALL
# ====================================
def hf_call(prompt: str):
    try:
        print("⏳ Calling Hugging Face...")
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=150
        )
        print("✅ Done")
        return response.choices[0].message.content.strip()
    except Exception as e:
        print("❌ ERROR:", e)
        return "Error generating response"

# ====================================
# 🔍 NODE 1: SEARCH
# ====================================
def search_web(state: ResearchState):
    print(f"\n🔍 Searching: {state['topic']}")

    new_results = [
        hf_call(f"Give a short informative snippet about {state['topic']}"),
        hf_call(f"Give another different snippet about {state['topic']}")
    ]

    return {
        "search_results": state["search_results"] + new_results,
        "steps_done": state["steps_done"] + 1
    }

# ====================================
# 📊 NODE 2: ANALYZE
# ====================================
def analyze_results(state: ResearchState):
    print(f"\n📊 Analyzing {len(state['search_results'])} results")

    joined = "\n".join(state["search_results"])

    analysis = hf_call(f"""
    Analyze these sources and extract key insights:

    {joined}

    Give 2-3 key points.
    """)

    return {
        "analysis": analysis,
        "steps_done": state["steps_done"] + 1
    }

# ====================================
# ✍️ NODE 3: SUMMARIZE
# ====================================
def summarize(state: ResearchState):
    print("\n✍️ Generating summary...")

    summary = hf_call(f"""
    Summarize this in simple terms:

    {state['analysis']}
    """)

    return {"summary": summary}

# ====================================
# 🔁 NODE 4: CONDITION
# ====================================
def should_continue(state: ResearchState):
    if len(state["search_results"]) < 3:
        return "search_web"
    return "summarize"

# ====================================
# 🔗 BUILD GRAPH
# ====================================
g = StateGraph(ResearchState)

g.add_node("search_web", search_web)
g.add_node("analyze", analyze_results)
g.add_node("summarize", summarize)

g.set_entry_point("search_web")

g.add_edge("search_web", "analyze")

g.add_conditional_edges(
    "analyze",
    should_continue,
    {
        "search_web": "search_web",
        "summarize": "summarize"
    }
)

g.add_edge("summarize", END)

# ====================================
# ▶️ RUN
# ====================================
if __name__ == "__main__":
    app = g.compile()

    result = app.invoke({
        "topic": "Quantum Computing",
        "search_results": [],
        "analysis": "",
        "summary": "",
        "steps_done": 0
    })

    print("\n✅ FINAL SUMMARY:\n")
    print(result["summary"])

Enter your Hugging Face API key: ··········
✅ API Loaded!

🔍 Searching: Quantum Computing
⏳ Calling Hugging Face...
✅ Done
⏳ Calling Hugging Face...
✅ Done

📊 Analyzing 2 results
⏳ Calling Hugging Face...
✅ Done

🔍 Searching: Quantum Computing
⏳ Calling Hugging Face...
✅ Done
⏳ Calling Hugging Face...
✅ Done

📊 Analyzing 4 results
⏳ Calling Hugging Face...
✅ Done

✍️ Generating summary...
⏳ Calling Hugging Face...
✅ Done

✅ FINAL SUMMARY:

**What is Quantum Computing?**

Quantum computers are powerful machines that can solve complex problems that regular computers can't. They work differently by using special units called qubits that can do many tasks at the same time.

**Challenges:**

1. Quantum computers are very delicate and can easily make mistakes.
2. Researchers have found ways to fix these mistakes, but it's still a big challenge.

**What can Quantum Computers do?**

1. They can solve problems that are too hard for regular computers.
2. They can look at many possibilities at th

# Scenario: AI-Powered Study Assistant (Flashcard-Based)
1. State Definition
The assistant maintains a notebook-like state for each learner:
- topic → The subject the learner is studying (e.g., "Photosynthesis").
- flashcard → A generated Q&A card created by Groq (question on one side, answer on the other).
- progress → A log of all past flashcards attempted, including whether the learner got them correct or not.

2. Workflow (Graph of States)
Each learner interaction flows through nodes until a flashcard is delivered:
- Input Node
- Learner provides a topic or asks for practice (e.g., "Test me on cell biology").
- State updates: topic = "cell biology"
- Generation Node (Groq API)
- Groq generates a flashcard:
- flashcard.question = "What organelle is known as the powerhouse of the cell?"
- flashcard.answer = "Mitochondria"
- Response Node
- Assistant presents the flashcard question to the learner.
- Evaluation Node
- Learner responds with their answer.
- Assistant checks correctness and updates progress.
- History Node
- Logs the flashcard attempt:
- progress = [{question, learner_answer, correct/incorrect}]

In [8]:
# ====================================
# Step 1: Install dependency
# ====================================
!pip install huggingface-hub --quiet

# ====================================
# Step 2: Import libraries
# ====================================
from huggingface_hub import InferenceClient
from getpass import getpass

# ====================================
# Step 3: Secure API Key
# ====================================
api_key = getpass("Enter your Hugging Face API key: ")

client = InferenceClient(api_key=api_key)

# ====================================
# 🧠 STATE (Notebook Memory)
# ====================================
state = {
    "topic": None,
    "flashcard": None,
    "progress": []
}

# ====================================
# 🤖 LLM CALL
# ====================================
def call_llm(prompt):
    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=150
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"ERROR: {str(e)}"

# ====================================
# 🔹 INPUT NODE
# ====================================
def input_node(user_input):
    state["topic"] = user_input
    print(f"\n📘 Topic set to: {state['topic']}")

# ====================================
# 🔹 GENERATION NODE (Flashcard Creator)
# ====================================
def generate_flashcard():
    prompt = f"""
    Generate a flashcard for topic: {state['topic']}

    Format strictly:
    Question: ...
    Answer: ...
    """

    result = call_llm(prompt)

    # Simple parsing
    try:
        question = result.split("Question:")[1].split("Answer:")[0].strip()
        answer = result.split("Answer:")[1].strip()
    except:
        question = "Error generating question"
        answer = "Error generating answer"

    state["flashcard"] = {
        "question": question,
        "answer": answer
    }

# ====================================
# 🔹 RESPONSE NODE
# ====================================
def present_question():
    print("\n🧠 Flashcard Question:")
    print("👉", state["flashcard"]["question"])

# ====================================
# 🔹 EVALUATION NODE
# ====================================
def evaluate_answer(user_answer):
    correct_answer = state["flashcard"]["answer"]

    is_correct = user_answer.lower().strip() in correct_answer.lower()

    result = "correct" if is_correct else "incorrect"

    print("\n✅ Correct!" if is_correct else "\n❌ Incorrect!")
    print("📌 Correct Answer:", correct_answer)

    return result, correct_answer

# ====================================
# 🔹 HISTORY NODE
# ====================================
def update_progress(user_answer, result, correct_answer):
    state["progress"].append({
        "question": state["flashcard"]["question"],
        "correct_answer": correct_answer,
        "user_answer": user_answer,
        "result": result
    })

# ====================================
# 🔹 MAIN LOOP
# ====================================
def run_study_assistant():
    print("\n🎓 AI Study Assistant Started!")

    while True:
        user_input = input("\nEnter topic (or 'exit'): ")

        if user_input.lower() == "exit":
            break

        # Step 1: Input Node
        input_node(user_input)

        while True:
            # Step 2: Generate Flashcard
            generate_flashcard()

            # Step 3: Ask Question
            present_question()

            # Step 4: User Answer
            user_answer = input("Your Answer (or 'stop'): ")

            if user_answer.lower() == "stop":
                break

            # Step 5: Evaluate
            result, correct_answer = evaluate_answer(user_answer)

            # Step 6: Update History
            update_progress(user_answer, result, correct_answer)

        print("\n📊 Progress so far:")
        for p in state["progress"]:
            print(p)

# ====================================
# 🚀 RUN
# ====================================
run_study_assistant()

Enter your Hugging Face API key: ··········

🎓 AI Study Assistant Started!

Enter topic (or 'exit'): photosynthesis

📘 Topic set to: photosynthesis

🧠 Flashcard Question:
👉 What is the process by which plants convert sunlight into energy?
Your Answer (or 'stop'): photosynthesis

✅ Correct!
📌 Correct Answer: Photosynthesis.

🧠 Flashcard Question:
👉 What is the primary function of photosynthesis in plants?
Your Answer (or 'stop'): stop

📊 Progress so far:
{'question': 'What is the process by which plants convert sunlight into energy?', 'correct_answer': 'Photosynthesis.', 'user_answer': 'photosynthesis', 'result': 'correct'}

Enter topic (or 'exit'): exit


# Scenario: AI-Powered Project Tracker (Task-Based)
1. State Definition
The assistant maintains a notebook-like state for each project:
- task → The specific work item or milestone (e.g., "Prepare Q1 financial report").
- status → The current state of the task (e.g., "in progress", "completed", "blocked").
- log → A history of all updates, including who made them and when.

2. Workflow (Graph of States)
Each project update flows through nodes until the task status is refreshed:
- Input Node
- Team member submits an update (e.g., "Report draft completed").
- State updates: task = "Q1 financial report"
- Processing Node (Groq API)
- Groq interprets the update and assigns a status:
- status = "completed"
- Response Node
- Assistant confirms the update back to the team:
- "Task Q1 financial report marked as completed."
- History Node
- Logs the update:
- log = [{task: "Q1 financial report", update: "draft completed", status: "completed", timestamp}]

In [10]:
# ====================================
# Step 1: Install dependency
# ====================================
!pip install huggingface-hub --quiet

# ====================================
# Step 2: Imports
# ====================================
from huggingface_hub import InferenceClient
from getpass import getpass
from datetime import datetime

# ====================================
# Step 3: Secure API Key
# ====================================
api_key = getpass("Enter your Hugging Face API key: ")
client = InferenceClient(api_key=api_key)

# ====================================
# 🧠 STATE (Project Memory)
# ====================================
state = {
    "task": None,
    "status": None,
    "log": []
}

# ====================================
# 🤖 LLM CALL
# ====================================
def call_llm(prompt):
    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=100
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"ERROR: {str(e)}"

# ====================================
# 🔹 INPUT NODE
# ====================================
def input_node(task_name, update_text):
    state["task"] = task_name
    print(f"\n📌 Task: {state['task']}")
    print(f"📝 Update: {update_text}")
    return update_text

# ====================================
# 🔹 PROCESSING NODE (LLM decides status)
# ====================================
def processing_node(update_text):
    prompt = f"""
    Given this update: "{update_text}"

    Classify task status into one of:
    - completed
    - in progress
    - blocked

    Return ONLY one word.
    """

    result = call_llm(prompt).lower()

    # Safety fallback
    if "complete" in result:
        status = "completed"
    elif "block" in result:
        status = "blocked"
    else:
        status = "in progress"

    state["status"] = status
    return status

# ====================================
# 🔹 RESPONSE NODE
# ====================================
def response_node():
    msg = f"✅ Task '{state['task']}' marked as {state['status']}."
    print("\n🤖 Assistant:", msg)
    return msg

# ====================================
# 🔹 HISTORY NODE
# ====================================
def history_node(update_text, user_name="User"):
    entry = {
        "task": state["task"],
        "update": update_text,
        "status": state["status"],
        "user": user_name,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    state["log"].append(entry)

# ====================================
# 🔹 MAIN SYSTEM
# ====================================
def run_project_tracker():
    print("\n🚀 AI Project Tracker Started!")

    # Ask task only once
    task_name = input("\nEnter task name (or 'exit'): ")
    if task_name.lower() == "exit":
        return

    state["task"] = task_name

    while True:
        update_text = input("\nEnter update (or 'change' / 'exit'): ")

        if update_text.lower() == "exit":
            break

        # Change task if needed
        if update_text.lower() == "change":
            task_name = input("Enter new task: ")
            state["task"] = task_name
            continue

        print(f"\n📌 Task: {state['task']}")
        print(f"📝 Update: {update_text}")

        # Processing
        processing_node(update_text)

        # Response
        response_node()

        # Log
        history_node(update_text)

        # Show logs
        print("\n📊 Project Log:")
        for log in state["log"]:
            print(log)

# ====================================
# 🚀 RUN
# ====================================
run_project_tracker()

Enter your Hugging Face API key: ··········

🚀 AI Project Tracker Started!

Enter task name (or 'exit'): Q1 financial report

Enter update (or 'change' / 'exit'): Draft completed

📌 Task: Q1 financial report
📝 Update: Draft completed

🤖 Assistant: ✅ Task 'Q1 financial report' marked as completed.

📊 Project Log:
{'task': 'Q1 financial report', 'update': 'Draft completed', 'status': 'completed', 'user': 'User', 'timestamp': '2026-03-20 07:08:31'}

Enter update (or 'change' / 'exit'): Waiting for approval

📌 Task: Q1 financial report
📝 Update: Waiting for approval

🤖 Assistant: ✅ Task 'Q1 financial report' marked as completed.

📊 Project Log:
{'task': 'Q1 financial report', 'update': 'Draft completed', 'status': 'completed', 'user': 'User', 'timestamp': '2026-03-20 07:08:31'}
{'task': 'Q1 financial report', 'update': 'Waiting for approval', 'status': 'completed', 'user': 'User', 'timestamp': '2026-03-20 07:12:29'}

Enter update (or 'change' / 'exit'): change
Enter new task: UI completed


 # Scenario: AI Symptom Tracker (Question-Based)
1. State Definition
The assistant maintains a notebook-like state for each patient:
- symptom → The patient’s reported issue (e.g., "persistent cough").
- observations → Notes or snippets generated by Groq about possible causes or related conditions.
- analysis → A synthesized interpretation of the observations.
- recommendation → A simplified, non-medical summary suggesting next steps (e.g., "consult a doctor if symptoms persist").
- steps_done → A counter tracking progress through the workflow.

2. Workflow (Graph of States)
Each patient query flows through nodes:
- Symptom Input Node
- Patient reports a symptom.
- State updates: symptom = "persistent cough"
- Observation Node (Groq API)
- Groq generates possible related factors or general information.
- Updates observations.
- Analysis Node
- Joins observations and extracts key insights.
- Updates analysis.
- Conditional Node
- If fewer than 3 observations are collected → loop back to Observation Node.
- If 3+ observations are available → move to Recommendation Node.
- Recommendation Node
- Generates a simplified, non-medical recommendation (e.g., "Seek medical advice if cough lasts more than 2 weeks").
- Updates recommendation.
- End Node
- Delivers the final recommendation to the patient.

In [13]:
# ====================================
# 🚀 STEP 1: Install Dependency
# ====================================
!pip install huggingface-hub --quiet

# ====================================
# 📦 STEP 2: Imports
# ====================================
from huggingface_hub import InferenceClient
from getpass import getpass

# ====================================
# 🔐 STEP 3: API KEY (Secure Input)
# ====================================
api_key = getpass("Enter your Hugging Face API key: ")

client = InferenceClient(api_key=api_key)

print("✅ API Loaded Successfully!")

# ====================================
# 🧠 STATE
# ====================================
state = {
    "symptom": None,
    "observations": [],
    "analysis": None,
    "recommendation": None,
    "steps_done": 0
}

# ====================================
# 🤖 LLM CALL (WITH DEBUG)
# ====================================
def call_llm(prompt):
    try:
        print("\n⏳ Calling LLM...")

        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=120
        )

        print("✅ Response received")
        return response.choices[0].message.content.strip()

    except Exception as e:
        print("❌ ERROR:", e)
        return "Error generating response"

# ====================================
# 🔹 INPUT NODE
# ====================================
def symptom_input():
    symptom = input("\nEnter your symptom (or 'exit'): ")

    if symptom.lower() == "exit":
        return None

    state["symptom"] = symptom
    return symptom

# ====================================
# 🔹 OBSERVATION NODE
# ====================================
def observation_node():
    print("\n[Observation Agent] Working...")

    prompt = f"""
    Symptom: {state['symptom']}

    Give ONE possible cause or related observation.
    Keep it short (1 line).
    """

    obs = call_llm(prompt)

    state["observations"].append(obs)
    state["steps_done"] += 1

    print("📌 Observation:", obs)

# ====================================
# 🔹 ANALYSIS NODE
# ====================================
def analysis_node():
    print("\n[Analysis Agent] Working...")

    prompt = f"""
    Symptom: {state['symptom']}

    Observations:
    {state['observations']}

    Give simple insights.
    """

    result = call_llm(prompt)
    state["analysis"] = result

    print("🧠 Analysis:", result)

# ====================================
# 🔹 RECOMMENDATION NODE
# ====================================
def recommendation_node():
    print("\n[Recommendation Agent] Working...")

    prompt = f"""
    Symptom: {state['symptom']}
    Analysis: {state['analysis']}

    Give a simple, non-medical recommendation.
    """

    result = call_llm(prompt)
    state["recommendation"] = result

    print("💡 Recommendation:", result)

# ====================================
# 🔁 MAIN SYSTEM
# ====================================
def run_symptom_tracker():
    print("\n🩺 AI Symptom Tracker Started!")

    while True:
        symptom = symptom_input()

        if symptom is None:
            print("👋 Exiting...")
            break

        # Reset state for new input
        state["observations"] = []
        state["analysis"] = None
        state["recommendation"] = None
        state["steps_done"] = 0

        print("\n🔄 Collecting observations...")

        # 🔁 Loop until 3 observations
        while len(state["observations"]) < 3:
            observation_node()

        # Analysis
        analysis_node()

        # Recommendation
        recommendation_node()

        print("\n✅ FINAL OUTPUT:")
        print(state["recommendation"])

# ====================================
# ▶️ RUN
# ====================================
run_symptom_tracker()

Enter your Hugging Face API key: ··········
✅ API Loaded Successfully!

🩺 AI Symptom Tracker Started!

Enter your symptom (or 'exit'): "persistent cough"

🔄 Collecting observations...

[Observation Agent] Working...

⏳ Calling LLM...
✅ Response received
📌 Observation: Pneumonia is a possible cause of a persistent cough.

[Observation Agent] Working...

⏳ Calling LLM...
✅ Response received
📌 Observation: Chronic bronchitis, a condition in which the airways are inflamed, often causing a persistent cough.

[Observation Agent] Working...

⏳ Calling LLM...
✅ Response received
📌 Observation: Chronic bronchitis or pneumonia could be a possible cause of a persistent cough.

[Analysis Agent] Working...

⏳ Calling LLM...
✅ Response received
🧠 Analysis: **Possible Causes of a Persistent Cough:**

1. **Pneumonia**: Inflammation of the lungs, which can cause a persistent cough.
2. **Chronic Bronchitis**: Long-term inflammation of the airways, leading to a persistent cough.
3. **Pneumonia or Chronic